# EfficientNetV2-S · Focal Loss + Camera Aug + SSL

Identyczna architektura co `EfficientNetV2_SSL_v2.ipynb`, ale `CrossEntropyLoss` zastąpiony przez **Focal Loss**.

| | |
|---|---|
| **Loss** | `FocalLoss(gamma=1.0, alpha=class_weights)` — skupia trening na trudnych przykładach |
| **Sampler** | `WeightedRandomSampler` (inverse-freq) — balansuje batche |
| **Fine-tuning** | 3-stage: head → last 3 blocks → full model |
| **SSL** | 2 rundy, progi 0.95 / 0.90 |
| **gamma** | `1.0` — bezpieczny start; zmień na `2.0` jeśli chcesz mocniej skupić się na trudnych |


In [7]:
import zipfile, os
ZIP = 'multi-view-pig-posture-recognition.zip'
if os.path.exists(ZIP) and not os.path.exists('multiview_pig_posture_recognition'):
    print('Extracting...')
    with zipfile.ZipFile(ZIP, 'r') as z:
        z.extractall('.')
    print('Done.')
else:
    print('Data already extracted or ZIP not found — skipping.')

import os; print(os.listdir('.'))

Data already extracted or ZIP not found — skipping.
['.claude', '.git', '.gitignore', 'augmentacja.ipynb', 'confidence_r1.png', 'confidence_shift.png', 'Convnext_Augmentacja.ipynb', 'Dino_Augmentacja.ipynb', 'eda_bbox_analysis.png', 'eda_centroids_per_class.png', 'eda_class_distribution.png', 'eda_crops_per_class.png', 'eda_domain_shift.png', 'eda_full_frames_cameras.png', 'eda_test_images.png', 'EfficientNetV2_FocalLoss.ipynb', 'EfficientNetV2_SAM_TTA.ipynb', 'EfficientNetV2_SSL.ipynb', 'EfficientNetV2_SSL_Progressive.ipynb', 'EfficientNetV2_SSL_Rotation.ipynb', 'EfficientNetV2_SSL_v2.ipynb', 'EfficientNetV2_YOLO_Seg.ipynb', 'EfficientNet_V2.ipynb', 'effnetv2s_ssl_final.pt', 'effnetv2s_supervised.pt', 'effnetv2_initial.pt', 'effnetv2_ssl_progressive.pt', 'EficientNet_Augmentacja.ipynb', 'full_training_summary.png', 'gpu_config.ipynb', 'multiview_pig_posture_recognition', 'Notebook.ipynb', 'phase1_curves.png', 'README.md', 'submission_ssl_progressive.csv', 'submission_v2.csv', 'T1_effi

In [8]:
import ast, copy, re, time
from collections import Counter

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
from torchvision import transforms, models
import torchvision.transforms.functional as TF

from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    x = torch.randn(10000, 10000, device='cuda')
    t0 = time.time(); _ = x @ x; torch.cuda.synchronize()
    print(f'GPU test: {time.time()-t0:.3f}s')
    del x; torch.cuda.empty_cache()

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
GPU test: 0.466s


In [9]:
CLASS_NAMES = {
    0: 'Lateral_lying_left',
    1: 'Lateral_lying_right',
    2: 'Sitting',
    3: 'Standing',
    4: 'Sternal_lying',
}
NUM_CLASSES = len(CLASS_NAMES)

BASE_DIR    = Path('multiview_pig_posture_recognition')
TRAIN2_IMGS = BASE_DIR / 'train2_images'
TEST_IMGS   = BASE_DIR / 'test_images'

BATCH_SIZE = 32

# ── Focal Loss ────────────────────────────────────────────────────────────────
FOCAL_GAMMA = 1.5   # 0 = identyczny z CE; 1 = łagodne skupienie; 2 = mocne skupienie

# ── 3-stage fine-tuning ───────────────────────────────────────────────────────
EPOCHS_S3  = 8;   LR_S3 = 5e-5

# ── SSL ───────────────────────────────────────────────────────────────────────
EPOCHS_SSL      = 5
SSL_CONF_THRESH = [0.95, 0.90]
LR_SSL          = [2e-5, 1e-5]

SAVE_SUPERVISED = 'effnetv2s_focal_supervised.pt'
SAVE_SSL_FINAL  = 'effnetv2s_focal_ssl_final.pt'
SUBMISSION_PATH = 'submission_focal.csv'

print(f'Focal gamma = {FOCAL_GAMMA}')
print(f'Epochs: S3={EPOCHS_S3} SSL={EPOCHS_SSL}x2')

Focal gamma = 1.5
Epochs: S3=8 SSL=5x2


## Focal Loss

```
FL(p_t) = -alpha_t · (1 − p_t)^γ · log(p_t)
```

- `p_t` — prawdopodobieństwo przypisane przez model do **poprawnej** klasy  
- Gdy model jest pewny i ma rację → `(1−p_t)^γ ≈ 0` → mały gradient (łatwe przykłady wyciszone)  
- Gdy model się myli lub waha → `(1−p_t)^γ ≈ 1` → pełny gradient (trudne przykłady wzmocnione)  
- `alpha_t` — wagi klas (takie same jak w `CrossEntropyLoss`), korygują nierównowagę klas

In [10]:
class FocalLoss(nn.Module):
    """
    Focal Loss dla klasyfikacji wieloklasowej.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Args:
        gamma:  focusing parameter (0 = identyczny z CE, zalecane: 1–2)
        alpha:  tensor wag klas (opcjonalny); ten sam format co
                weight w nn.CrossEntropyLoss
    """
    def __init__(self, gamma: float = 1.0, alpha: torch.Tensor = None):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            # register_buffer: tensor podróżuje razem z modelem na GPU/CPU
            self.register_buffer('alpha', alpha.float())
        else:
            self.alpha = None

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits:  (B, C) — surowe wyjście modelu (przed softmax)
            targets: (B,)   — indeksy klas
        Returns:
            scalar — uśredniony focal loss po batchu
        """
        # 1. Standardowy CE bez redukcji, opcjonalnie z wagami klas
        ce = F.cross_entropy(logits, targets,
                             weight=self.alpha,
                             reduction='none')          # shape: (B,)

        # 2. p_t = exp(-CE) = prawdopodobieństwo poprawnej klasy
        pt = torch.exp(-ce)                             # shape: (B,)

        # 3. Focal weight: wycisza łatwe (pt ≈ 1), wzmacnia trudne (pt ≈ 0)
        focal_weight = (1.0 - pt) ** self.gamma        # shape: (B,)

        # 4. Finalny loss
        focal = focal_weight * ce                      # shape: (B,)
        return focal.mean()

    def extra_repr(self) -> str:
        return f'gamma={self.gamma}, alpha={"tak" if self.alpha is not None else "brak"}'


# ── Szybki test poprawności ───────────────────────────────────────────────────
with torch.no_grad():
    _logits  = torch.randn(8, NUM_CLASSES)
    _targets = torch.randint(0, NUM_CLASSES, (8,))
    _ce_val  = F.cross_entropy(_logits, _targets).item()
    _fl_val  = FocalLoss(gamma=0.0)(_logits, _targets).item()  # gamma=0 ≡ CE
    _fl2_val = FocalLoss(gamma=1.0)(_logits, _targets).item()
    assert abs(_ce_val - _fl_val) < 1e-4, 'FocalLoss(gamma=0) != CE!'
    assert _fl2_val <= _ce_val, 'gamma>0 powinno dać loss <= CE'
    print(f'CE = {_ce_val:.4f}  |  FL(γ=0) = {_fl_val:.4f}  |  FL(γ=1) = {_fl2_val:.4f}  ✓')
print(f'FocalLoss OK  (gamma={FOCAL_GAMMA})')

CE = 1.9555  |  FL(γ=0) = 1.9555  |  FL(γ=1) = 1.6854  ✓
FocalLoss OK  (gamma=1.5)


In [11]:
# ── Wczytanie danych ──────────────────────────────────────────────────────────
def parse_camera_meta(image_id):
    m = re.match(r'(pen\d+)_(orb|tur)_(cam\d+)_', str(image_id))
    return (m.group(1), m.group(2), m.group(3)) if m else ('unknown', 'unknown', 'unknown')

def add_camera_cols(df):
    df['pen']      = df['image_id'].apply(lambda x: parse_camera_meta(x)[0])
    df['cam_type'] = df['image_id'].apply(lambda x: parse_camera_meta(x)[1])
    df['cam_num']  = df['image_id'].apply(lambda x: parse_camera_meta(x)[2])
    df['camera']   = df['pen'] + '_' + df['cam_type'] + '_' + df['cam_num']
    return df

train2 = pd.read_csv(BASE_DIR / 'train2.csv')
train2['source']      = 'train2'
train2['bbox_parsed'] = train2['bbox'].apply(ast.literal_eval)
train2['class_name']  = train2['class_id'].map(CLASS_NAMES)
train2 = add_camera_cols(train2)

test = pd.read_csv(BASE_DIR / 'test.csv')
test['source']      = 'test'
test['bbox_parsed'] = test['bbox'].apply(ast.literal_eval)
test = add_camera_cols(test)

print(f'Train2: {len(train2):,} instancji  |  {train2["image_id"].nunique():,} zdjęć')
print(f'Test:   {len(test):,} instancji   |  {test["image_id"].nunique():,} zdjęć')
print('\nRozkład klas (train2):')
for cname, cnt in train2['class_name'].value_counts().items():
    print(f'  {cname:25s}: {cnt:,}')

Train2: 23,450 instancji  |  3,150 zdjęć
Test:   11,708 instancji   |  1,350 zdjęć

Rozkład klas (train2):
  Standing                 : 9,928
  Sternal_lying            : 6,309
  Lateral_lying_right      : 3,435
  Lateral_lying_left       : 3,083
  Sitting                  : 695


In [12]:
# ── Narzędzia obrazu ──────────────────────────────────────────────────────────
def load_image(image_id, source):
    folder = {'train2': TRAIN2_IMGS, 'test': TEST_IMGS}[source]
    return Image.open(folder / image_id).convert('RGB')

def crop_with_padding(image, bbox, padding=0.12, make_square=True):
    img_w, img_h = image.size
    x, y, w, h   = map(float, bbox)
    x1 = x - w * padding;      y1 = y - h * padding
    x2 = x + w + w * padding;  y2 = y + h + h * padding
    if make_square:
        side   = max(x2 - x1, y2 - y1)
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        x1, x2 = cx - side / 2, cx + side / 2
        y1, y2 = cy - side / 2, cy + side / 2
    x1 = max(0, int(round(x1)));      y1 = max(0, int(round(y1)))
    x2 = min(img_w, int(round(x2)));  y2 = min(img_h, int(round(y2)))
    return image.crop((x1, y1, max(x2, x1+1), max(y2, y1+1)))

In [13]:
# ── Augmentacje per kamera ────────────────────────────────────────────────────
#
# Transformacje PIL symulujące wygląd kamer testowych na podstawie kamer treningowych.
# Flipa poziomy wymaga zamiany etykiet:
#   Lateral_lying_left (0) <-> Lateral_lying_right (1)
#   klasy 2-4 symetryczne — etykieta bez zmian

FLIP_LABEL_MAP = {0: 1, 1: 0, 2: 2, 3: 3, 4: 4}

def aug_pen2_tur_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_saturation(img, saturation_factor=0.8)
    return TF.adjust_brightness(img, brightness_factor=0.95)

def aug_pen1_tur_cam2(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=1.35)
    return TF.adjust_saturation(img, saturation_factor=0.9)

def aug_pen2_orb_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=0.6)
    img = TF.adjust_contrast(img, contrast_factor=1.5)
    arr = np.array(img).astype(float)
    arr[:,:,0] = (arr[:,:,0] * 0.85).clip(0, 255)
    arr[:,:,1] = (arr[:,:,1] * 1.10).clip(0, 255)
    arr[:,:,2] = (arr[:,:,2] * 0.80).clip(0, 255)
    shifted = Image.fromarray(arr.clip(0,255).astype(np.uint8))
    grey    = np.array(TF.to_grayscale(shifted, num_output_channels=3)).astype(float)
    return Image.fromarray((0.65*arr + 0.35*grey).clip(0,255).astype(np.uint8))

def aug_pen2_tur_cam2(img):
    img = TF.adjust_brightness(img, brightness_factor=1.3)
    return TF.adjust_saturation(img, saturation_factor=0.85)

def aug_pen2_orb_cam2(img):
    img  = TF.adjust_brightness(img, brightness_factor=0.60)
    img  = TF.adjust_contrast(img, contrast_factor=1.5)
    arr  = np.array(img).astype(float)
    grey = np.array(TF.to_grayscale(img, num_output_channels=3)).astype(float)
    return Image.fromarray((0.65*arr + 0.35*grey).clip(0,255).astype(np.uint8))

CAMERA_AUG_FN = {
    'pen2_tur_cam1': (aug_pen2_tur_cam1, True),
    'pen1_tur_cam2': (aug_pen1_tur_cam2, True),
    'pen2_orb_cam1': (aug_pen2_orb_cam1, True),
    'pen2_tur_cam2': (aug_pen2_tur_cam2, False),
    'pen2_orb_cam2': (aug_pen2_orb_cam2, False),
}

for cam, (_, flip) in CAMERA_AUG_FN.items():
    print(f'  {cam:20s}  flip={flip}')

  pen2_tur_cam1         flip=True
  pen1_tur_cam2         flip=True
  pen2_orb_cam1         flip=True
  pen2_tur_cam2         flip=False
  pen2_orb_cam2         flip=False


In [14]:
# ── Transformacje ─────────────────────────────────────────────────────────────
class AddGaussianNoise:
    def __init__(self, std=0.02, p=0.15):
        self.std, self.p = std, p
    def __call__(self, t):
        if torch.rand(1).item() < self.p:
            t = torch.clamp(t + torch.randn_like(t) * self.std, 0., 1.)
        return t

MU  = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

BASE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(MU, STD),
])

VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MU, STD),
])

SSL_TRANSFORM = transforms.Compose([
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    transforms.RandomRotation(degrees=10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(MU, STD),
])

In [15]:
# ── Klasy datasetów ───────────────────────────────────────────────────────────
class PigDatasetCameraAug(Dataset):
    def __init__(self, df, camera_aug_fn, base_transform, is_train=True):
        self.camera_aug_fn  = camera_aug_fn
        self.base_transform = base_transform
        df = df.reset_index(drop=True)
        self.df = df
        self.samples = []
        for idx, row in df.iterrows():
            cam   = row.get('camera', 'unknown')
            label = int(row['class_id'])
            self.samples.append((idx, False, label))
            if is_train and cam in camera_aug_fn:
                _, does_flip = camera_aug_fn[cam]
                self.samples.append((idx, True,
                                     FLIP_LABEL_MAP[label] if does_flip else label))

    def __len__(self):  return len(self.samples)

    def __getitem__(self, i):
        row_i, do_aug, label = self.samples[i]
        row  = self.df.iloc[row_i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'])
        if do_aug:
            aug_fn, _ = self.camera_aug_fn[row['camera']]
            crop = aug_fn(crop)
        return self.base_transform(crop), label


class PigTestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):  return len(self.df)
    def __getitem__(self, i):
        row  = self.df.iloc[i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'])
        return self.transform(crop), str(row['row_id'])


class PseudoLabelDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):  return len(self.df)
    def __getitem__(self, i):
        row  = self.df.iloc[i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'])
        return self.transform(crop), int(row['class_id'])


print('Dataset classes ✓')

Dataset classes ✓


In [16]:
# ── Pomocniki: wagi klas + DataLoader ─────────────────────────────────────────
def compute_focal_alpha(labels):
    """
    Wagi klas (alpha) dla FocalLoss: √(inverse-frequency), unormowane.
    Rzadkie klasy (np. Sitting) dostają wyższe wagi → silniejszy gradient.
    Pierwiastek łagodzi korektę względem pełnego inverse-frequency.
    """
    counts  = np.bincount(np.asarray(labels, dtype=int), minlength=NUM_CLASSES)
    inv_f   = 1.0 / np.maximum(counts, 1)
    alpha   = np.sqrt(inv_f / inv_f.mean())   # √ inverse-freq, unormowane
    return torch.FloatTensor(alpha).to(DEVICE)


def build_weighted_loader(dataset, labels, batch_size, num_workers=0):
    labels = np.asarray(labels, dtype=int)
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    w      = (1.0 / np.maximum(counts, 1))[labels]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(w), num_samples=len(w), replacement=True
    )
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=num_workers, pin_memory=True)


# ── Dataset treningowy ────────────────────────────────────────────────────────
train_ds     = PigDatasetCameraAug(train2, CAMERA_AUG_FN, BASE_TRANSFORM)
train_labels = [s[2] for s in train_ds.samples]
train_loader = build_weighted_loader(train_ds, train_labels, BATCH_SIZE)

focal_alpha  = compute_focal_alpha(train_labels)   # alpha dla FocalLoss

counts_aug = np.bincount(train_labels, minlength=NUM_CLASSES)
print(f'Próbki po camera-aug: {len(train_ds):,}  |  batchy: {len(train_loader)}')
print('\nRozkład klas + alfa Focal Loss:')
for cid, (cnt, a) in enumerate(zip(counts_aug, focal_alpha.cpu())):
    bar = '█' * int(20 * cnt / counts_aug.max())
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:>6,}  alpha={a:.3f}  {bar}')

Próbki po camera-aug: 44,490  |  batchy: 1391

Rozkład klas + alfa Focal Loss:
  [0] Lateral_lying_left       :  6,272  alpha=0.815  ██████
  [1] Lateral_lying_right      :  6,060  alpha=0.830  ██████
  [2] Sitting                  :  1,358  alpha=1.752  █
  [3] Standing                 : 19,137  alpha=0.467  ████████████████████
  [4] Sternal_lying            : 11,663  alpha=0.598  ████████████


In [17]:
# ── Model + zamrażanie warstw ─────────────────────────────────────────────────
def create_model():
    m = models.efficientnet_v2_s(
        weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    )
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    return m.to(DEVICE)

def freeze_backbone(model):
    for p in model.features.parameters():   p.requires_grad = False
    for p in model.classifier.parameters(): p.requires_grad = True

def unfreeze_last_n_blocks(model, n=3):
    for p in model.parameters(): p.requires_grad = False
    for block in list(model.features.children())[-n:]:
        for p in block.parameters(): p.requires_grad = True
    for p in model.classifier.parameters(): p.requires_grad = True

def unfreeze_all(model):
    for p in model.parameters(): p.requires_grad = True

def n_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = create_model()
print(f'EfficientNetV2-S  |  {sum(p.numel() for p in model.parameters()):,} parametrów')

EfficientNetV2-S  |  20,183,893 parametrów


In [18]:
# ── Pętla treningowa ──────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, preds_all, targets_all = 0.0, [], []
    n = len(loader)
    for bi, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds_all.extend(out.argmax(1).detach().cpu().tolist())
        targets_all.extend(labels.cpu().tolist())
        bar = '█' * int(30*(bi+1)/n) + '░' * (30 - int(30*(bi+1)/n))
        print(f'\r  |{bar}| {100*(bi+1)/n:5.1f}%  loss={loss.item():.4f}',
              end='', flush=True)
    print()
    return (total_loss / len(loader.dataset),
            accuracy_score(targets_all, preds_all),
            f1_score(targets_all, preds_all, average='macro', zero_division=0))


def run_stage(model, loader, criterion, optimizer, scheduler,
              n_epochs, stage_name, history, best_f1, best_state):
    print(f'\n{"─"*65}')
    print(f'{stage_name}  |  trenuję: {n_trainable(model):,} parametrów')
    print(f'{"─"*65}')
    for epoch in range(n_epochs):
        t0 = time.time()
        loss, acc, f1 = train_one_epoch(model, loader, optimizer, criterion)
        if scheduler: scheduler.step()
        elapsed = time.time() - t0
        history.append({'stage': stage_name, 'epoch': len(history)+1,
                        'train_loss': loss, 'train_acc': acc, 'train_f1': f1})
        flag = '  ← best' if f1 > best_f1 else ''
        print(f'  Ep {epoch+1:>2}/{n_epochs} ({elapsed/60:.1f}m)  '
              f'loss={loss:.4f}  acc={acc:.4f}  f1={f1:.4f}  '
              f'lr={optimizer.param_groups[0]["lr"]:.1e}{flag}', flush=True)
        if f1 > best_f1:
            best_f1    = f1
            best_state = copy.deepcopy(model.state_dict())
    return best_f1, best_state


def generate_pseudo_labels(model, test_df, transform):
    loader = DataLoader(PigTestDataset(test_df, transform),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)
    preds_all, confs_all = [], []
    model.eval()
    with torch.no_grad():
        for bi, (imgs, _) in enumerate(loader):
            probs        = F.softmax(model(imgs.to(DEVICE)), dim=1)
            confs, preds = probs.max(dim=1)
            preds_all.extend(preds.cpu().tolist())
            confs_all.extend(confs.cpu().tolist())
            bar = '█' * int(20*(bi+1)/len(loader)) + '░' * (20-int(20*(bi+1)/len(loader)))
            print(f'\r  [pseudo] |{bar}| {bi+1}/{len(loader)}', end='', flush=True)
    print()
    result = test_df.copy().reset_index(drop=True)
    result['class_id']   = preds_all
    result['confidence'] = confs_all
    return result


def plot_history(history_list, vlines=None, title='', fname=None):
    df = pd.DataFrame(history_list)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for col, ax, color, ylabel in [
        ('train_loss', axes[0], 'steelblue', 'Focal Loss'),
        ('train_f1',   axes[1], 'green',     'Macro F1'),
    ]:
        ax.plot(df['epoch'], df[col], marker='o', markersize=3, color=color)
        if vlines:
            for vx, lbl in vlines:
                ax.axvline(vx, color='gray', linestyle=':', alpha=0.7, label=lbl)
            ax.legend(fontsize=8)
        ax.set_xlabel('Epoka'); ax.set_ylabel(ylabel); ax.grid(alpha=0.3)
    best = df['train_f1'].max()
    axes[1].axhline(best, color='red', linestyle='--', linewidth=1, label=f'best={best:.4f}')
    axes[1].legend()
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if fname: plt.savefig(fname, dpi=130, bbox_inches='tight')
    plt.show()
    return best


print('Narzędzia treningowe ✓')

Narzędzia treningowe ✓


## Faza 1 — Trening nadzorowany (3-etapowe odmrażanie)

In [19]:
# ── Kryterium: Focal Loss z alfa per klasa ────────────────────────────────────
criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=focal_alpha)
print(f'Kryterium: {criterion}')
print('Alfa (wagi klas):')
for cid, a in enumerate(focal_alpha.cpu()):
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {a:.4f}')

Kryterium: FocalLoss(gamma=1.5, alpha=tak)
Alfa (wagi klas):
  [0] Lateral_lying_left       : 0.8155
  [1] Lateral_lying_right      : 0.8296
  [2] Sitting                  : 1.7525
  [3] Standing                 : 0.4668
  [4] Sternal_lying            : 0.5980


In [ ]:
history_sup = []
best_f1_sup, best_state_sup = 0.0, None

# Etap 3 — pełny model, cosine LR
unfreeze_all(model)
opt_s3   = torch.optim.Adam(model.parameters(), lr=LR_S3)
sched_s3 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_s3, T_max=EPOCHS_S3, eta_min=1e-6
)
best_f1_sup, best_state_sup = run_stage(
    model, train_loader, criterion, opt_s3, scheduler=sched_s3,
    n_epochs=EPOCHS_S3, stage_name='Etap 3 — pełny model (cosine)',
    history=history_sup, best_f1=best_f1_sup, best_state=best_state_sup
)

model.load_state_dict(best_state_sup)
torch.save({'model_state_dict': best_state_sup,
            'class_names':      CLASS_NAMES,
            'best_train_f1':    best_f1_sup,
            'focal_gamma':      FOCAL_GAMMA,
            'phase':            'supervised'}, SAVE_SUPERVISED)
print(f'\nFaza 1 najlepszy macro-F1: {best_f1_sup:.4f}')
print(f'Zapisano → {SAVE_SUPERVISED}')




─────────────────────────────────────────────────────────────────
Etap 3 — pełny model (cosine)  |  trenuję: 20,183,893 parametrów
─────────────────────────────────────────────────────────────────
  |█████████████████████████░░░░░|  83.8%  loss=0.0573

In [ ]:
plot_history(
    history_sup,
    title=f'Faza 1 — Focal Loss (γ={FOCAL_GAMMA})',
    fname='phase1_focal_curves.png'
)

## Faza 2 — SSL (progi 0.95 / 0.90)

In [ ]:
ckpt = torch.load(SAVE_SUPERVISED, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Wczytano model Fazy 1  (F1={ckpt["best_train_f1"]:.4f})')


def run_ssl_round(model, train2_df, pseudo_df, epochs, lr, round_num,
                  global_ep_offset, history_acc):
    print(f'\n{"="*65}')
    print(f'SSL Runda {round_num}  |  próg={SSL_CONF_THRESH[round_num-1]}  '
          f'pseudo={len(pseudo_df):,}  lr={lr}')
    print(f'{"="*65}')

    sup_ds    = PigDatasetCameraAug(train2_df, CAMERA_AUG_FN, BASE_TRANSFORM)
    pseudo_ds = PseudoLabelDataset(pseudo_df, SSL_TRANSFORM)
    combined  = ConcatDataset([sup_ds, pseudo_ds])

    all_labels = [s[2] for s in sup_ds.samples] + pseudo_df['class_id'].astype(int).tolist()

    # Focal Loss przeliczony na połączonym zbiorze (nowe proporcje klas)
    ssl_alpha     = compute_focal_alpha(all_labels)
    criterion_ssl = FocalLoss(gamma=FOCAL_GAMMA, alpha=ssl_alpha)
    loader        = build_weighted_loader(combined, all_labels, BATCH_SIZE)

    combined_counts = np.bincount(all_labels, minlength=NUM_CLASSES)
    print(f'  Nadzorowane (z cam-aug): {len(sup_ds):,}')
    print(f'  Pseudo-etykiety:         {len(pseudo_ds):,}')
    print(f'  Łącznie:                 {len(combined):,}  |  {len(loader)} batchy')
    print('  Rozkład klas + alfa:')
    for cid, (cnt, a) in enumerate(zip(combined_counts, ssl_alpha.cpu())):
        print(f'    [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:>6,}  alpha={a:.3f}')

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_f1, best_state = 0.0, None

    for ep in range(epochs):
        t0 = time.time()
        loss, acc, f1 = train_one_epoch(model, loader, optimizer, criterion_ssl)
        global_ep_offset += 1
        history_acc.append({'stage': f'SSL-R{round_num}', 'epoch': global_ep_offset,
                            'train_loss': loss, 'train_acc': acc, 'train_f1': f1})
        flag = '  ← best' if f1 > best_f1 else ''
        print(f'  R{round_num} Ep {ep+1:>2}/{epochs} ({(time.time()-t0)/60:.1f}m)  '
              f'loss={loss:.4f}  acc={acc:.4f}  f1={f1:.4f}{flag}', flush=True)
        if f1 > best_f1:
            best_f1    = f1
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print(f'  Runda {round_num} najlepszy F1: {best_f1:.4f}')
    return model, best_f1, global_ep_offset


print('Funkcja SSL ✓')

In [ ]:
# ── SSL Runda 1 (próg 0.95) ───────────────────────────────────────────────────
print('Generuję pseudo-etykiety (model Fazy 1)...')
pseudo_all_r1  = generate_pseudo_labels(model, test, VAL_TRANSFORM)
pseudo_high_r1 = pseudo_all_r1[pseudo_all_r1['confidence'] >= SSL_CONF_THRESH[0]].copy()

print(f'Próg {SSL_CONF_THRESH[0]}: '
      f'{len(pseudo_high_r1):,} / {len(pseudo_all_r1):,} '
      f'({100*len(pseudo_high_r1)/len(pseudo_all_r1):.1f}%)')
for cid, cnt in sorted(Counter(pseudo_high_r1['class_id'].tolist()).items()):
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:,}')

plt.figure(figsize=(10, 4))
plt.hist(pseudo_all_r1['confidence'], bins=50, edgecolor='black', alpha=0.75, color='steelblue')
plt.axvline(SSL_CONF_THRESH[0], color='red',    linestyle='--', label=f'R1={SSL_CONF_THRESH[0]}')
plt.axvline(SSL_CONF_THRESH[1], color='orange', linestyle='--', label=f'R2={SSL_CONF_THRESH[1]}')
plt.xlabel('Pewność predykcji'); plt.ylabel('Liczba instancji')
plt.title('Rozkład pewności — model Fazy 1')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('confidence_r1.png', dpi=120, bbox_inches='tight')
plt.show()

history_ssl = []
global_ep   = len(history_sup)

model, ssl_f1_r1, global_ep = run_ssl_round(
    model, train2, pseudo_high_r1,
    epochs=EPOCHS_SSL, lr=LR_SSL[0],
    round_num=1, global_ep_offset=global_ep, history_acc=history_ssl
)

In [ ]:
# ── SSL Runda 2 (próg 0.90, zaktualizowany model) ─────────────────────────────
print('Generuję zaktualizowane pseudo-etykiety (model Rundy 1)...')
pseudo_all_r2  = generate_pseudo_labels(model, test, VAL_TRANSFORM)
pseudo_high_r2 = pseudo_all_r2[pseudo_all_r2['confidence'] >= SSL_CONF_THRESH[1]].copy()

print(f'Próg {SSL_CONF_THRESH[1]}: '
      f'{len(pseudo_high_r2):,} / {len(pseudo_all_r2):,} '
      f'({100*len(pseudo_high_r2)/len(pseudo_all_r2):.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, pall, ttl in [
    (axes[0], pseudo_all_r1, f'Model Fazy 1  (próg={SSL_CONF_THRESH[0]})'),
    (axes[1], pseudo_all_r2, f'Model Rundy 1  (próg={SSL_CONF_THRESH[1]})'),
]:
    ax.hist(pall['confidence'], bins=50, edgecolor='black', alpha=0.75, color='steelblue')
    ax.set_title(ttl); ax.set_xlabel('Pewność'); ax.grid(alpha=0.3)
plt.suptitle('Przesunięcie rozkładu pewności po SSL Rundzie 1', fontweight='bold')
plt.tight_layout()
plt.savefig('confidence_shift.png', dpi=120, bbox_inches='tight')
plt.show()

model, ssl_f1_r2, global_ep = run_ssl_round(
    model, train2, pseudo_high_r2,
    epochs=EPOCHS_SSL, lr=LR_SSL[1],
    round_num=2, global_ep_offset=global_ep, history_acc=history_ssl
)

In [ ]:
# ── Zapis finalnego modelu ────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names':      CLASS_NAMES,
    'focal_gamma':      FOCAL_GAMMA,
    'phase1_f1':        best_f1_sup,
    'ssl_round1_f1':    ssl_f1_r1,
    'ssl_round2_f1':    ssl_f1_r2,
    'phase':            'ssl_final',
}, SAVE_SSL_FINAL)

print(f'Zapisano → {SAVE_SSL_FINAL}')
print(f'  Faza 1 F1:    {best_f1_sup:.4f}')
print(f'  SSL Runda 1:  {ssl_f1_r1:.4f}')
print(f'  SSL Runda 2:  {ssl_f1_r2:.4f}')


In [ ]:
# ── Zbiorczy wykres treningu ──────────────────────────────────────────────────
s3e =  EPOCHS_S3
plot_history(
    history_sup + history_ssl,
    vlines=[
        (s3e+0.5, 'E3→SSL'), (s3e+EPOCHS_SSL+0.5, 'R1→R2'),
    ],
    title=f'Pełny trening — Focal Loss (γ={FOCAL_GAMMA}) + SSL (0.95/0.90)',
    fname='full_focal_training.png'
)

## Inferencja i plik submisji

In [ ]:
ckpt = torch.load(SAVE_SSL_FINAL, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Wczytano finalny model  (SSL R2 F1={ckpt["ssl_round2_f1"]:.4f})')

test_loader = DataLoader(
    PigTestDataset(test, VAL_TRANSFORM),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True
)

all_row_ids, all_preds = [], []
with torch.no_grad():
    for bi, (imgs, row_ids) in enumerate(test_loader):
        preds = model(imgs.to(DEVICE)).argmax(dim=1).cpu().tolist()
        all_row_ids.extend(list(row_ids))
        all_preds.extend(preds)
        bar = '█' * int(30*(bi+1)/len(test_loader)) + '░' * (30-int(30*(bi+1)/len(test_loader)))
        print(f'\r  |{bar}| {bi+1}/{len(test_loader)}', end='', flush=True)

print(f'\n{len(all_preds):,} predykcji')

submission = pd.DataFrame({'row_id': all_row_ids, 'class_id': all_preds})
sample_sub = pd.read_csv(BASE_DIR / 'sample_submission.csv')

assert set(submission['row_id']) == set(sample_sub['row_id']), 'Niezgodne row_id!'
assert submission['class_id'].between(0, 4).all(), 'Błędne class_id!'

submission = (
    submission.set_index('row_id')
              .reindex(sample_sub['row_id'])
              .reset_index()
)

print('\nRozkład predykcji:')
for cid, cnt in sorted(submission['class_id'].value_counts().items()):
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:,}  ({100*cnt/len(submission):.1f}%)')

submission.to_csv(SUBMISSION_PATH, index=False)
print(f'\nZapisano → {SUBMISSION_PATH}')


# ── instead of files.download(), just print where the files are ──────────────
print(f'\nModel saved to:      {os.path.abspath(SAVE_SSL_FINAL)}')
print(f'Submission saved to: {os.path.abspath(SUBMISSION_PATH)}')
print('Done ✓')